In [1]:
from sedona.spark import SedonaContext
from IPython.display import Image, display

from sedona.spark import dataframe_to_arrow
import geopandas as gpd
from sedona.spark.stats.weighting import add_distance_band_column
from sedona.spark.maps.SedonaKepler import SedonaKepler
import os

from slope import get_average_curvature

s


In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/30 22:36:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/30 22:36:08 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/11/30 22:36:08 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/11/30 22:36:08 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/11/30 22:36:08 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/11/30 22:36:08 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/11/30 22:36:08 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import FloatType
import pandas as pd
import pyspark.sql.functions as f

germany_polygon = """POLYGON((
  8.9765 47.2701,
  13.8396 47.2701,
  13.8396 50.5646,
  8.9765 50.5646,
  8.9765 47.2701
))""".replace("\n", "").replace("  ", " ")

connector = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/connector")
transportation = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/transportation").repartition(100)
infrastructure = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/infrastructure").\
    where(f"ST_Intersects(geometry, ST_GeomFromText('{germany_polygon}'))")
places = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/places")
accidents = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/accidents/")

In [7]:
min_promiles = accidents.selectExpr("MIN(promiles)").collect()[0][0]
max_promiles = accidents.selectExpr("MAX(promiles)").collect()[0][0]
num_buckets = 10
bucket_size = (max_promiles - min_promiles) / num_buckets  # = 0.3

In [ ]:
accidents = (
    accidents.withColumn(
        "promiles",
        f.when(f.col("promiles") < min_promiles, 0)  # values below range
         .when(f.col("promiles") >= max_promiles, num_buckets - 1)  # values above range
         .otherwise(f.floor((f.col("promiles") - min_promiles) / bucket_size))
    )
)

In [6]:
values_mapping = {
    "accident_category": {
        "1": "fatal",
        "2": "major_injuries",
        "3": "minor_injuries"
    },
    "lighting_condition": {
        "0": "Daylight",
        "1": "Twilight",
        "2": "Darkness"
    }
}

In [39]:
h3_cells = sedona.sql(
    f"""
    WITH h3_cells AS (
        SELECT
            id,
            ST_H3ToGeom(ARRAY(id))[0] AS geom
        LATERAL VIEW EXPLODE(ST_H3CellIDs(ST_GeomFromText('{germany_polygon}'), 8, true)) AS id
    )
    SELECT 
        id,
        ST_Transform(geom, 'epsg:4326', 'epsg:3044') AS geometry
    FROM h3_cells
    """
)

In [40]:
h3_cells.count()

188952

In [11]:
SedonaKepler.create_map(accidents.selectExpr("ST_Transform(geometry, 'epsg:3044', 'epsg:4326')").limit(10000))

/usr/local/lib/python3.10/dist-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string
                                                                                

KeplerGl(data={'unnamed': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,…

In [12]:
# Create weight dataframe

In [41]:
from sedona.spark.stats.hotspot_detection.getis_ord import g_local
import pyspark.sql.functions as f
from sedona.spark.stats.weighting import add_distance_band_column

In [42]:
accidents_agg = h3_cells.alias("h").\
    join(accidents.alias("a"), on=f.expr("ST_Intersects(h.geometry, a.geometry)")).\
    groupBy("h.id").\
    agg(f.count("*").alias("value"))

In [43]:
accidents_agg.count()

25780

In [44]:
accidents_agg_values = h3_cells.join(accidents_agg.select("id", "value"), on="id", how="left").\
    selectExpr("id", "geometry", "Coalesce(value, 0) AS value").repartition(40)

In [45]:
accidents_agg_values.cache().count()

188952

In [46]:
weights_df = add_distance_band_column(
    dataframe=accidents_agg_values,
    threshold=1000,
    include_self=True
)

In [47]:
from sedona.stats.hotspot_detection.getis_ord import g_local

gi_df = g_local(
    dataframe=weights_df,
    x="value",
    star=True
)

In [48]:
gi_df.cache().count()

188952

In [49]:
munich_area_wkt = """
POLYGON((
  10.438908 47.667611,
  12.663354 47.667611,
  12.663354 48.685899,
  10.438908 48.685899,
  10.438908 47.667611
))
""".replace("\n", "").replace(". ", " ")

df = gi_df.where(
    f"ST_Intersects(geometry, ST_Transform(ST_GeomFromText('{munich_area_wkt}'), 'epsg:4326', 'epsg:3044'))"
).select("id", "Z", "P", "geometry")

In [50]:
gi_df.show()

+------------------+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                id|            geometry|value|             weights|                   G|                  EG|                  VG|                   Z|                   P|
+------------------+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|613044535011835903|POLYGON ((565395....|    1|[{{61304453481470...|4.589963280293757...|6.880054193657648E-5|9.028768056731883E-9|-0.24101189383232025|  0.4047729491301554|
|613046617062244351|POLYGON ((535956....|    1|[{{61304661707273...|3.518971848225214...|6.880054193657648E-5|9.028768056731883E-9|   2.979342769379046|0.001444337189403...|
|613046686769479679|POLYGON ((641761....|    0|[{{61304668803196...|3.059975520195838...|6.880054193657648E-5|9.028768056731883E-9

In [51]:
from sedona.spark import SedonaKepler
import json

with open("hotspot_viz_color.json") as f:
    config_hotspot_color = json.load(f)


In [52]:
SedonaKepler.create_map(df, "map", config=config_hotspot_color)

KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [], 'layers': [{'id': 'ddoefy', 'type': '…

# basic statistics

In [53]:
import pyspark.sql.functions as f
accidents_category_mapping_expr = f.create_map(
    [f.lit(x) for pair in values_mapping["accident_category"].items() for x in pair]
)

lighting_condition_category_mapping_expr = f.create_map(
    [f.lit(x) for pair in values_mapping["lighting_condition"].items() for x in pair]
)

aggregated_accidents = accidents.groupBy("accident_category", "lighting_condition").\
    count().\
    orderBy(f.col("accident_category")).\
    select(
        accidents_category_mapping_expr[f.col("accident_category")].alias("category"),
        lighting_condition_category_mapping_expr[f.col("lighting_condition")].alias("lighting_condition"),
        f.col("count")
    )

In [54]:
summarized = aggregated_accidents.groupBy("lighting_condition").sum().\
    selectExpr("lighting_condition", "`sum(count)` AS count")

In [55]:
aggregated_accidents.alias("a").join(summarized.alias("s"), on="lighting_condition").\
    selectExpr("category", "lighting_condition", "ROUND(a.count/s.count * 100, 2) AS percent").\
    orderBy(f.col("category"), f.desc("percent")).\
    show()

[Stage 181:>                                                      (0 + 10) / 10]

+--------------+------------------+-------+
|      category|lighting_condition|percent|
+--------------+------------------+-------+
|         fatal|          Darkness|   1.35|
|         fatal|          Twilight|   1.05|
|         fatal|          Daylight|   0.96|
|major_injuries|          Darkness|  17.28|
|major_injuries|          Daylight|  16.95|
|major_injuries|          Twilight|  16.73|
|minor_injuries|          Twilight|  82.22|
|minor_injuries|          Daylight|  82.09|
|minor_injuries|          Darkness|  81.37|
+--------------+------------------+-------+



# Feature engineering

In [56]:
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import FloatType, BooleanType
import pandas as pd
from shapely.geometry.linestring import LineString
from sedona.spark.sql.functions import sedona_vectorized_udf
import math

german_speed_limits = {
    'motorway': 130,
    'trunk': 100,
    'primary': 80,
    'secondary': 80,
    'tertiary': 50,
    'residential': 30
}

typical_widths = {
    'motorway': 25,
    'trunk': 20,
    'primary': 12,
    'secondary': 10,
    'tertiary': 8,
    'residential': 6,
    'service': 5,
}

widths_mapping_expr = f.create_map(
    [f.lit(x) for pair in typical_widths.items() for x in pair]
)

@pandas_udf(FloatType())
def map_speed_limit(rss: pd.Series,rcs: pd.Series) -> pd.Series:
    def first_non_empty(road_speed, road_cls):
        default = german_speed_limits[road_cls]
        if road_speed is None:
            return default
            
        for element in road_speed:
            max_speed = element.get("max_speed")
            if not max_speed:
                continue
            speed_value = max_speed.get("value")
            
            if speed_value:
                return speed_value
                
        return default

    df = pd.DataFrame({'road_class': rcs, 'road_speed': rss})
        
    return df.apply(lambda x: first_non_empty(
        x["road_speed"], x["road_class"]
    ), axis=1)


def match_road_flag(flag_value: str):
    @pandas_udf(BooleanType())
    def is_road_flag(road_flags: pd.Series) -> pd.Series:
        def contains_road_flag(flags: list):
            if flags is None:
                return False

            for flag in flags:
                if flag_value in flag["values"]:
                    return True

            return False

        return road_flags.apply(lambda x: contains_road_flag(x))

    return is_road_flag

@sedona_vectorized_udf(return_type=FloatType())
def get_average_curvature_udf(line: LineString):
    return get_average_curvature(line)

In [57]:
crossings = infrastructure.where("class == 'crossing'").\
    selectExpr(
        "id",
        "ST_Transform(geometry, 'epsg:4326', 'epsg:3044') AS geometry"
    )

In [58]:
SedonaKepler.create_map(crossings.limit(10_000), "crossings")

KeplerGl(data={'crossings': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2…

In [71]:
cycleways = transportation.where("class == 'cycleway'").\
    select("id", "geometry")

is_bridge = match_road_flag("is_bridge")(
    f.col("road_flags")
).alias("is_bridge")

is_tunnel = match_road_flag("is_tunnel")(
    f.col("road_flags")
).alias("is_tunnel")

is_under_construction = match_road_flag("is_under_construction")(
    f.col("road_flags")
).alias("is_under_construction")

road_classes = ('motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'residential')

enriched_transportation = (
    transportation
        .where(f"class in {road_classes}")
        .withColumn(
            "geometry",
            f.expr("ST_Transform(geometry, 'epsg:4326', 'epsg:3044')"))
        .withColumn(
            "width", 
            widths_mapping_expr[f.col("class")])
        .withColumn(
            "max_speed",
            map_speed_limit(f.col("speed_limits"), f.col("class")))
        .select(
            "class",
            "width",
            "id",
            "road_flags",
            "geometry",
            "width",
            "max_speed",
            is_bridge,
            is_tunnel,
            is_under_construction,
            get_average_curvature_udf("geometry").alias("curvature"),
            "quality"
        )
)

In [72]:
roads_with_accidents = enriched_transportation.alias("t").\
    join(
        accidents.alias("a"),
        on=f.expr("ST_DWithin(t.geometry, a.geometry, t.width)")
    ).selectExpr("t.*", "a.lighting_condition", "a.accident_category", "a.promiles")

In [73]:
roads_with_accidents.count()

290270

In [74]:
roads_with_accidents.cache().count()

290270

In [75]:
roads_feature_values = roads_with_accidents.alias("ra").\
    join(
        crossings.alias("c"),
        on=f.expr("ST_KNN(ra.geometry, c.geometry, 1)")
    ).\
    selectExpr(
        "ra.id",
        "ra.geometry",
        "ra.max_speed AS ms",
        "ra.is_bridge AS is_b",
        "ra.is_tunnel AS is_t",
        "ra.is_under_construction AS is_uc",
        "ROUND(ra.curvature, 3) AS c",
        "ra.lighting_condition AS lc",
        "ra.accident_category AS ac",
        "ROUND(ST_Distance(c.geometry, ra.geometry), 2) AS distance",
        "ra.promiles",
        "ra.quality"
    )

In [76]:
roads_feature_values.cache().count()

290270

In [77]:
roads_feature_values.show(10)

+--------------------+--------------------+-----+-----+-----+-----+-----+---+---+--------+--------+-------+
|                  id|            geometry|   ms| is_b| is_t|is_uc|    c| lc| ac|distance|promiles|quality|
+--------------------+--------------------+-----+-----+-----+-----+-----+---+---+--------+--------+-------+
|0861fa80efffffff0...|LINESTRING (57995...|120.0| true|false|false|0.018|  0|  2|  198.37|       0|      3|
|0861fa80efffffff0...|LINESTRING (57995...|120.0| true|false|false|0.018|  0|  2|  198.37|       2|      3|
|0861fa80efffffff0...|LINESTRING (57995...|120.0| true|false|false|0.018|  1|  3|  198.37|       0|      3|
|0861fa80efffffff0...|LINESTRING (57995...|120.0| true|false|false|0.018|  2|  3|  198.37|       0|      3|
|0891fa8c9dcbffff0...|LINESTRING (51487...| 30.0|false|false|false|  0.0|  0|  3|  272.12|       4|      3|
|0861fa8d37ffffff0...|LINESTRING (50489...| 70.0|false|false|false|0.051|  2|  3| 2377.32|       5|      5|
|0861fa8d37ffffff0...|LINEST

In [78]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("id").orderBy(f.desc("count"))
similar_size_buckets = Window.partitionBy("ac").orderBy("id")

# each road can have more than one accident, we group them by id, accident category, lighting condition and by the promiles in blood range
# than for our training model we take the most popular bucket for this grouping
# to get similar size bucket based on the accident category we use again window function and we use ordering by id to always get the same input
# we take buckets less than 1000
# that's our input, you can play with different approaches like
# - selecting different grouping columns
# - creating variables based on for instance accident category, so you would have 
# -----------------------------------------------------
# | id | fatal_avg_promiles | fatal_most_common_lc | major_injuries_avg_promiles | .....
roads_feature_values_ranked = roads_feature_values.\
    groupBy("id", "ac", "lc", "promiles").\
    agg(
        f.col("id"),
        f.col("promiles"),
        f.col("ac"),
        f.col("lc"),
        f.count("*").alias("count"),
        f.first("ms").alias("ms"),
        f.first("is_b").alias("is_b"),
        f.first("is_t").alias("is_t"),
        f.first("is_uc").alias("is_uc"),
        f.first("c").alias("c"),
        f.first("distance").alias("distance"),
        f.first("quality").alias("quality"),
    ).\
    withColumn("rank", row_number().over(window_spec)).\
    where("rank == 1").\
    withColumn("bucket_size", row_number().over(similar_size_buckets)).\
    where("bucket_size < 1000").\
    drop("rank", "bucket_size")

In [452]:
roads_feature_values_ranked.cache().count()

2678

# Modelling

In [79]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
)
from pyspark.sql.functions import col

# Step 2: Convert booleans to integers
roads_feature_values.cache().count()

roads_feature_values_transformed = roads_feature_values_ranked.\
    withColumn("is_b", col("is_b").cast("int")).\
    withColumn("is_t", col("is_t").cast("int")).\
    withColumn("is_uc", col("is_uc").cast("int"))

In [80]:
def index_string_column(column_name: str) -> list:
    indexer = StringIndexer(
        inputCol=column_name,
        outputCol=f"{column_name}_index"
    )
    
    encoder = OneHotEncoder(
        inputCol=f"{column_name}_index",
        outputCol=f"{column_name}_encoded"
    )

    return [indexer, encoder]

def min_max_scale_column(column_name: str) -> list:
    ms_assembler = VectorAssembler(
        inputCols=[column_name],
        outputCol=f"{column_name}_vec"
    )
    
    ms_scaler = MinMaxScaler(
        inputCol=f"{column_name}_vec",
        outputCol=f"{column_name}_scaled"
    )

    return [ms_assembler, ms_scaler]

# Step 5: Convert label 'ac' to numeric
label_indexer = StringIndexer(
    inputCol="ac",
    outputCol="label"
)

# Step 6: Create vector 
assembler = VectorAssembler(
    inputCols=[
        "lc_encoded",
        "ms_scaled",
        "is_b",
        "is_t",
        "is_uc",
        "c",
        "distance_scaled",
        "promiles_scaled",
        "quality_encoded"
    ],
    outputCol="features"
)

# Step 7: Compose pipeline
pipeline = Pipeline(stages=[
    *index_string_column("lc"),
    *index_string_column("quality"),
    label_indexer,
    *min_max_scale_column("distance"),
    *min_max_scale_column("promiles"),
    *min_max_scale_column("ms"),
    assembler,
])

# Fit and transform
model = pipeline.fit(roads_feature_values_transformed)
data_for_modelling = model.transform(roads_feature_values_transformed)

In [81]:
from xgboost.spark import SparkXGBClassifier

xgb_classifier = SparkXGBClassifier(
  features_col="features",
  label_col="label",
  num_workers=10,
)

In [82]:
train_df, test_df = data_for_modelling.randomSplit([0.8, 0.2], seed=42)

In [83]:
xgboost_model = xgb_classifier.fit(train_df)

2025-11-26 21:28:02,116 INFO XGBoost-PySpark: _fit Running xgboost-3.1.2 on 10 workers with
	booster params: {'objective': 'multi:softprob', 'device': 'cpu', 'num_class': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2025-11-26 21:28:05,282 INFO XGBoost-PySpark: _train_booster Training on CPUs10]
[21:28:06] Task 0 got rank 0[21:28:06] Task 1 got rank 1

[21:28:06] Task 2 got rank 2
[21:28:06] Task 4 got rank 4
[21:28:06] Task 3 got rank 3
[21:28:06] Task 5 got rank 5
[21:28:06] Task 9 got rank 9
[21:28:06] Task 6 got rank 6
[21:28:06] Task 7 got rank 7
[21:28:06] Task 8 got rank 8
[21:28:06] [0]	training-mlogloss:0.87822
[21:28:06] [1]	training-mlogloss:0.75241
[21:28:06] [2]	training-mlogloss:0.66643
[21:28:06] [3]	training-mlogloss:0.60606
[21:28:06] [4]	training-mlogloss:0.56181
[21:28:06] [5]	training-mlogloss:0.52904
[21:28:06] [6]	training-mlogloss:0.50245
[21:28:07] [7]	training-mlogloss

In [84]:
predictions = xgboost_model.transform(test_df)

In [85]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")

2025-11-26 21:28:13,426 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2025-11-26 21:28:14,034 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


Accuracy: 0.7336
F1 Score: 0.7338
Recall: 0.7336


2025-11-26 21:28:14,445 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
